In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
import json
with open("../citation.json", 'r') as f:
    citation = json.load(f)

In [3]:
citation

{'doi': 'https://doi.org/10.1016/j.devcel.2020.05.010',
 'citation': 'Shami, A.N., Zheng, X., Munyoki, S.K., Ma, Q., Manske, G.L., Green, C.D., Sukhwani, M., Orwig, K.E., Li, J.Z. and Hammoud, S.S., 2020. Single-cell RNA sequencing of human, macaque, and mouse testes uncovers conserved and divergent features of mammalian spermatogenesis. Developmental cell, 54(4), pp.529-547.',
 'table': 'https://ars.els-cdn.com/content/image/1-s2.0-S1534580720303993-mmc2.xlsx'}

## Define the metadata

In [4]:
organism = "homo_sapiens"
cell_source = "testis"
cell_state = None
table_name = "clean_degs.xlsx" # local name

## Read in file

In [5]:
df = pd.read_excel(table_name, sheet_name=2, skiprows=4)

df

,cluster,gene,p_val,avg_logFC,pct.1,pct.2,p_val_adj,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,Tcell,CD52,2.252593e-96,2.635621,0.738,0.005,1.048379e-91,NaN,NaN,NaN,NaN,NaN
1,Tcell,CD69,1.115023e-73,2.521717,0.646,0.008,5.189428e-69,NaN,NaN,NaN,NaN,NaN
2,Tcell,B2M,1.789482e-72,2.396398,0.969,0.255,8.328428e-68,NaN,NaN,NaN,NaN,NaN
3,Tcell,CCL5,1.105175e-70,2.855041,0.492,0.003,5.143596e-66,NaN,NaN,NaN,NaN,NaN
4,Tcell,CXCR4,9.216814e-63,2.380985,0.631,0.014,4.289598e-58,NaN,NaN,CellType,nCell,nMarker
...,...,...,...,...,...,...,...,...,...,...,...,...
3030,Elongating,RP11-456O19.4,1.176020e-166,0.717516,0.369,0.130,5.473317e-162,NaN,NaN,NaN,NaN,NaN
3031,Elongating,USPL1,8.613636e-166,0.708896,0.380,0.137,4.008872e-161,NaN,NaN,NaN,NaN,NaN
3032,Elongating,GOLM1,1.416608e-164,0.698258,0.361,0.127,6.593036e-160,NaN,NaN,NaN,NaN,NaN
3033,Elongating,AFAP1,1.769970e-161,0.702949,0.374,0.146,8.237617e-157,NaN,NaN,NaN,NaN,NaN


## Unfiltered

In [6]:
unfiltered_df = df.copy()
unfiltered_df.rename(columns ={"cluster": "cell_type_label", "gene": "feature_name"}, inplace=True)
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None

unfiltered_df.drop(['p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj', "Unnamed: 7", "Unnamed: 8", "Unnamed: 9", "Unnamed: 10", "Unnamed: 11"], axis=1, inplace=True) 
result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

## Save unfiltered

In [7]:
with open("evidence_unfiltered.json", 'w') as f:
    json.dump(result, f, indent = 4)

## Perform the filtering

In [8]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "cluster"
col_rank = "pct.1"

In [9]:
min_mean = 15
max_pval = 0.05
min_lfc = 1.5
max_gene_shares = 10
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")


# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [10]:
markers[col_ct].value_counts()

cluster
ImmLeydig         20
Endothelial       20
f-Pericyte        20
Elongating        20
Spermatogonia     20
Myoid             20
Tcell             20
Macrophage        20
m-Pericyte        20
Spermatocyte      14
RoundSpermatid     5
Name: count, dtype: int64

In [11]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cluster
RoundSpermatid    0.654800
Spermatocyte      0.712143
f-Pericyte        0.755750
Endothelial       0.767850
ImmLeydig         0.785800
Spermatogonia     0.815050
Myoid             0.825550
Tcell             0.865250
Elongating        0.883050
Macrophage        0.892150
m-Pericyte        0.899050
Name: pct.1, dtype: float64

## Find Ensembl ID

In [12]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [13]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=3):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [14]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json



In [15]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = df["feature_name"]
df["feature_identifier"] = df["feature_identifier"].apply(run)

result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

In [16]:
result

[{'cell_type_label': 'RoundSpermatid',
  'gene': 'TTN',
  'organism': 'homo_sapiens',
  'cell_source': 'testis',
  'cell_state': None,
  'gene_id': 'ENSG00000155657'},
 {'cell_type_label': 'RoundSpermatid',
  'gene': 'RP11-166P13.4',
  'organism': 'homo_sapiens',
  'cell_source': 'testis',
  'cell_state': None,
  'gene_id': 'gene not found'},
 {'cell_type_label': 'Spermatocyte',
  'gene': 'LYAR',
  'organism': 'homo_sapiens',
  'cell_source': 'testis',
  'cell_state': None,
  'gene_id': 'ENSG00000145220'},
 {'cell_type_label': 'ImmLeydig',
  'gene': 'IGF1',
  'organism': 'homo_sapiens',
  'cell_source': 'testis',
  'cell_state': None,
  'gene_id': 'ENSG00000017427'},
 {'cell_type_label': 'Endothelial',
  'gene': 'XAF1',
  'organism': 'homo_sapiens',
  'cell_source': 'testis',
  'cell_state': None,
  'gene_id': 'ENSG00000132530'},
 {'cell_type_label': 'Endothelial',
  'gene': 'TM4SF1',
  'organism': 'homo_sapiens',
  'cell_source': 'testis',
  'cell_state': None,
  'gene_id': 'ENSG00000

## Save evidence

In [17]:
with open("evidence.json", "w") as f:
    json.dump(result, f, indent = 4) 